# Eldercare SFT (QLoRA) from GGUF on Kaggle

Notebook này train tiếp mô hình từ file GGUF bằng QLoRA với dữ liệu eldercare.

In [ ]:
!pip install -q -U \
  "transformers>=4.46.0" \
  "datasets" \
  "accelerate" \
  "peft" \
  "trl" \
  "bitsandbytes" \
  "protobuf==5.29.5"

In [ ]:
import os, json
from glob import glob
import torch
from datasets import load_dataset, concatenate_datasets
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
)
from peft import LoraConfig
from trl import SFTTrainer

MODEL_DIR = '/kaggle/input/models/trnminhv/qwen3-gguf/pytorch/default/1'
DATA_DIR = '/kaggle/input/datasets/trnminhv/emotion-respone/eldercare_sft_final'
OUTPUT_DIR = '/kaggle/working/eldercare_qwen3_gguf_qlora'
os.makedirs(OUTPUT_DIR, exist_ok=True)

print('CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

gguf_files = glob(os.path.join(MODEL_DIR, '*.gguf'))
assert len(gguf_files) > 0, f'Không tìm thấy file .gguf trong {MODEL_DIR}'
GGUF_FILE = gguf_files[0]
print('Using GGUF:', GGUF_FILE)

In [ ]:
# Load tokenizer + model từ GGUF
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR,
    gguf_file=os.path.basename(GGUF_FILE),
    trust_remote_code=True,
    device_map='auto',
    quantization_config=bnb_config,
)
model.config.use_cache = False
print('Model loaded.')

In [ ]:
# Load dataset train/val/test
train_path = os.path.join(DATA_DIR, 'train.jsonl')
val_path = os.path.join(DATA_DIR, 'val.jsonl')
test_path = os.path.join(DATA_DIR, 'test.jsonl')

train_ds = load_dataset('json', data_files=train_path, split='train')
val_ds = load_dataset('json', data_files=val_path, split='train')
test_ds = load_dataset('json', data_files=test_path, split='train')

print('Train:', len(train_ds), 'Val:', len(val_ds), 'Test:', len(test_ds))

In [ ]:
# Format về text theo chat template
def to_text(example):
    text = tokenizer.apply_chat_template(
        example['messages'],
        tokenize=False,
        add_generation_prompt=False,
    )
    return {'text': text}

train_text = train_ds.map(to_text, remove_columns=train_ds.column_names)
val_text = val_ds.map(to_text, remove_columns=val_ds.column_names)
test_text = test_ds.map(to_text, remove_columns=test_ds.column_names)

print(train_text[0]['text'][:500])

In [ ]:
# Cấu hình QLoRA
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
    target_modules=[
        'q_proj', 'k_proj', 'v_proj', 'o_proj',
        'gate_proj', 'up_proj', 'down_proj'
    ]
)

train_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=2,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    warmup_ratio=0.05,
    lr_scheduler_type='cosine',
    logging_steps=20,
    save_strategy='epoch',
    eval_strategy='epoch',
    bf16=torch.cuda.is_available(),
    fp16=not torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False,
    gradient_checkpointing=True,
    report_to='none',
    max_grad_norm=1.0,
    seed=42,
)

In [ ]:
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_text,
    eval_dataset=val_text,
    peft_config=peft_config,
    args=train_args,
    dataset_text_field='text',
    max_seq_length=1024,
    packing=False,
)

trainer.train()
trainer.save_model(os.path.join(OUTPUT_DIR, 'adapter'))
tokenizer.save_pretrained(os.path.join(OUTPUT_DIR, 'adapter'))
print('Saved adapter to', os.path.join(OUTPUT_DIR, 'adapter'))

In [ ]:
# Test nhanh trên tập test (sample nhỏ)
samples = test_ds.select(range(min(5, len(test_ds))))
for i, ex in enumerate(samples):
    prompt = tokenizer.apply_chat_template(
        ex['messages'][:1], tokenize=False, add_generation_prompt=True
    )
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=120, do_sample=False)
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    print(f'\n===== SAMPLE {i+1} =====')
    print('USER:', ex['messages'][0]['content'])
    print('MODEL:', text[len(prompt):].strip())